# 02 - Netflix-paper weights and hidden units on MovieLens

## 1. Objective

Notebook 01 rewrote MovieLens ratings into the paper’s visible matrix.

$$
V \in \{0,1\}^{K \times m}
$$

The goal of this notebook is to move from visible units to hidden units.
Following the Netflix paper, we will define the weight tensor, hidden biases, and the hidden conditional probability.
This notebook stops at hidden activation probabilities and does not yet train the RBM.

## 2. Paper logic: hidden features as latent preference factors

In the paper, hidden units represent latent features or latent preference factors.
Each hidden unit is binary.
The hidden layer is shared across users through shared weights, even though each user has a different visible matrix.
Intuitively, the hidden layer summarizes patterns in the user’s observed ratings.

Given a user-specific visible matrix

$$
V
$$

the hidden layer converts the observed rating pattern into a lower-dimensional set of latent feature activations.

## 3. Paper equations relevant for weights and hidden units

### Eq. (2) Hidden conditional distribution

$$
p(h_j = 1 \mid V) =
\sigma\left(
 b_j +
 \sum_{i=1}^{m}
 \sum_{k=1}^{K}
 v_i^k W_{ij}^k
\right)
$$

Explanation of symbols:
- h_j is the j-th hidden unit.
- b_j is the hidden bias.
- v_i^k indicates whether movie i is observed at rating level k.
- W_{ij}^k is the weight linking movie i, rating level k, and hidden unit j.

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

$$
a_j =
 b_j +
 \sum_{i=1}^{m}
 \sum_{k=1}^{K}
 v_i^k W_{ij}^k
$$

$$
p(h_j = 1 \mid V) = \sigma(a_j)
$$

## 4. Weight structure in the Netflix paper

The visible matrix has shape K by m for one user.
The hidden layer has F units.
Therefore, the weights for one user-specific visible block can be viewed as a tensor of shape K by m by F.

In the paper’s notation, the index i refers to movies, not just arbitrary column positions.
Therefore the weight associated with a movie should conceptually be tied to that movie identity.
In implementation, once the movies for a user are ordered, the visible columns can be matched to the corresponding movie-specific slice of the weight tensor.

**The hidden-layer computation depends on movie identity, rating level, and hidden feature index simultaneously.**

## 5. Set model dimensions for this notebook

This notebook uses a small hidden dimension for interpretability and demonstration; larger values can be used later during training experiments.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

rating_levels = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
K = len(rating_levels)
F = 8

## 6. Reuse the visible-matrix example from Notebook 01

This section rebuilds the same user-specific visible matrix using the same data-loading logic and selection rule.

In [2]:
def resolve_project_root() -> Path:
    root = Path.cwd().resolve()
    if root.name == "notebooks":
        root = root.parent
    return root


def load_ratings_table() -> tuple[pd.DataFrame, Path]:
    root = resolve_project_root()
    candidates = [
        root / "data" / "processed" / "train_ratings.csv",
        root / "data" / "processed" / "ratings.csv",
        root / "data" / "processed" / "rating.csv",
        root / "data" / "rating.csv",
        root / "data" / "ratings.csv",
    ]

    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            if {"userId", "movieId", "rating"}.issubset(df.columns):
                return df[["userId", "movieId", "rating"]].copy(), path

    data_dir = root / "data"
    search_dirs = [data_dir / "processed", data_dir]
    for directory in search_dirs:
        if not directory.exists():
            continue
        for path in sorted(directory.glob("*.csv")):
            try:
                df = pd.read_csv(path)
            except Exception:
                continue
            if {"userId", "movieId", "rating"}.issubset(df.columns):
                return df[["userId", "movieId", "rating"]].copy(), path

    raise FileNotFoundError("No ratings table found with columns userId, movieId, rating.")


def build_visible_matrix_for_user(user_ratings_df: pd.DataFrame, rating_levels: list[float]):
    df = user_ratings_df[["movieId", "rating"]].copy()
    df = df.sort_values("movieId").reset_index(drop=True)

    ordered_movie_ids = df["movieId"].tolist()
    ordered_ratings = [round(float(r), 1) for r in df["rating"].tolist()]

    rating_to_index = {round(r, 1): idx for idx, r in enumerate(rating_levels)}
    K = len(rating_levels)
    m = len(df)

    V = np.zeros((K, m), dtype=int)
    for col, rating in enumerate(ordered_ratings):
        if rating not in rating_to_index:
            raise ValueError(f"Rating {rating} not in rating_levels vocabulary.")
        row = rating_to_index[rating]
        V[row, col] = 1

    return V, ordered_movie_ids, ordered_ratings, rating_to_index


ratings_df, ratings_path = load_ratings_table()
user_counts = ratings_df.groupby("userId").size().reset_index(name="count")
eligible_users = user_counts[user_counts["count"] >= 20]
eligible_users = eligible_users.sort_values(["count", "userId"], ascending=[False, True])

if eligible_users.empty:
    raise ValueError("No user found with at least 20 ratings.")

selected_user_id = int(eligible_users.iloc[0]["userId"])
user_ratings = ratings_df[ratings_df["userId"] == selected_user_id].copy()

V, ordered_movie_ids, ordered_ratings, rating_to_index = build_visible_matrix_for_user(
    user_ratings, rating_levels
)

m = V.shape[1]

## 7. Initialize weights and hidden biases

At this stage these are demonstration parameters only. They are not yet learned from data.

In [3]:
rng = np.random.default_rng(42)
W = rng.normal(loc=0.0, scale=0.01, size=(K, m, F))
b_h = np.zeros((F,), dtype=float)

## 8. Compute hidden pre-activations from V and W from Eq.(2)

$$
a_j =
 b_j +
 \sum_{i=1}^{m}
 \sum_{k=1}^{K}
 v_i^k W_{ij}^k
$$

The computation below uses these tensor shapes:
- V has shape (K, m)
- W has shape (K, m, F)

In [4]:
def compute_hidden_preactivation(V: np.ndarray, W: np.ndarray, b_h: np.ndarray) -> np.ndarray:
    if V.ndim != 2 or W.ndim != 3:
        raise ValueError("V must be 2D and W must be 3D.")
    if V.shape[0] != W.shape[0] or V.shape[1] != W.shape[1]:
        raise ValueError("V and W must align on K and m dimensions.")
    if W.shape[2] != b_h.shape[0]:
        raise ValueError("W hidden dimension must match b_h length.")

    hidden_sum = (V[:, :, None] * W).sum(axis=(0, 1))
    return b_h + hidden_sum


hidden_preactivation = compute_hidden_preactivation(V, W, b_h)

## 9. Convert pre-activations into hidden probabilities

The sampled binary hidden state is shown only for intuition. At this stage the main object of interest is the activation probability vector.

In [5]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


hidden_probs = sigmoid(hidden_preactivation)
hidden_state_sampled = rng.binomial(1, hidden_probs)

## 10. Display and interpret the result

In [ ]:
print("Selected user:", selected_user_id)
print("K rating levels:", K)
print("m rated movies:", m)
print("F hidden units:", F)
print("V shape:", V.shape)
print("W shape:", W.shape)
print("Hidden bias shape:", b_h.shape)
print("Hidden pre-activation shape:", hidden_preactivation.shape)
print("Hidden probability shape:", hidden_probs.shape)

preview_n = min(5, m)
print("First few ordered_movie_ids:", ordered_movie_ids[:preview_n])
print("First few ordered_ratings:", ordered_ratings[:preview_n])
print("Hidden pre-activation values:", hidden_preactivation)
print("Hidden probabilities:", hidden_probs)
print("Sampled hidden states:", hidden_state_sampled)

hidden_units = [f"h{idx+1}" for idx in range(F)]
summary_df = pd.DataFrame(
    {
        "hidden_unit": hidden_units,
        "preactivation": hidden_preactivation,
        "probability": hidden_probs,
        "sampled_state": hidden_state_sampled,
    }
)
summary_df

Selected user: 118205
K: 10
m: 7677
F: 8
V shape: (10, 7677)
W shape: (10, 7677, 8)
Hidden bias shape: (8,)
Hidden pre-activation shape: (8,)
Hidden probability shape: (8,)
First few ordered_movie_ids: [1, 2, 3, 4, 5]
First few ordered_ratings: [4.0, 4.0, 3.0, 3.0, 3.0]
Hidden pre-activation values: [-0.65211083  0.89068464 -0.53746774 -0.94997345 -1.61719711  2.01175918
  1.49636889 -0.01687314]
Hidden probabilities: [0.34251402 0.70903144 0.36877685 0.27889016 0.16559179 0.8820262
 0.81703228 0.49578181]
Sampled hidden states: [1 1 1 0 0 1 1 1]


,hidden_unit,preactivation,probability,sampled_state
0,h1,-0.652111,0.342514,1
1,h2,0.890685,0.709031,1
2,h3,-0.537468,0.368777,1
3,h4,-0.949973,0.278890,0
4,h5,-1.617197,0.165592,0
5,h6,2.011759,0.882026,1
6,h7,1.496369,0.817032,1
7,h8,-0.016873,0.495782,1


## 11. Tiny interpretation

Each hidden unit is a latent feature detector.
A larger activation probability means that the current observed rating pattern aligns more strongly with that latent feature.
At this stage the activations are not yet meaningful in a semantic sense because the weights are randomly initialized.
After learning, these hidden units can be interpreted as learned preference features.

## 12. End-of-notebook summary

1. The Netflix-paper visible matrix

$$
V
$$

has now been connected to a hidden layer through the weight tensor

$$
W
$$

2. The key hidden-side object is the activation probability computed from the visible ratings, weights, and hidden bias.

$$
p(h_j = 1 \mid V)
$$

3. The next notebook will introduce learning or parameter updates so that these hidden features become data-driven rather than randomly initialized.